In [1]:
from mini_competition.data_functions import load_data, add_zero_sales, split_train_set, write_result, PROCCESED_PATH, EXTRACTED_PATH, PROCCESED_PATH
import pandas as pd
import numpy as np

data = load_data(PROCCESED_PATH / "train.parquet" )
data = data[data["date"] > "2016-07-01"]

data.head()

,date,store_nbr,item_nbr,unit_sales,onpromotion
83757599,2016-07-02,1,103520,2.0,False
83757600,2016-07-02,1,103665,4.0,False
83757601,2016-07-02,1,105574,2.0,False
83757602,2016-07-02,1,105575,7.0,False
83757603,2016-07-02,1,105577,3.0,False


In [2]:
# add zero sales lines
data = add_zero_sales(data)

data.info()

<class 'pandas.DataFrame'>
RangeIndex: 89139096 entries, 0 to 89139095
Data columns (total 5 columns):
 #   Column       Dtype         
---  ------       -----         
 0   date         datetime64[ms]
 1   store_nbr    int8          
 2   item_nbr     int32         
 3   unit_sales   float32       
 4   onpromotion  object        
dtypes: datetime64[ms](1), float32(1), int32(1), int8(1), object(1)
memory usage: 2.1+ GB


In [3]:
item_data = load_data(EXTRACTED_PATH / "items.csv")
store_data = load_data(EXTRACTED_PATH / "stores.csv")
oil_data = load_data(EXTRACTED_PATH / "oil.csv" )
holiday_data = load_data(EXTRACTED_PATH / "holidays_events.csv" )

def add_item_data(d : pd.DataFrame, items : pd.DataFrame) -> pd.DataFrame:
    return  d.merge(items, on="item_nbr", how="left").copy()

def add_store_data(d :pd.DataFrame, stores: pd.DataFrame) -> pd.DataFrame:
    return  d.merge(stores, on="store_nbr", how="left").copy()

def add_holidays(d: pd.DataFrame, holidays: pd.DataFrame) -> pd.DataFrame:
    h = holidays.copy()
    h["date"] = pd.to_datetime(h["date"])
    national = h[(h["locale"] == "National") & (~h["transferred"])]["date"]

    one_day = pd.Timedelta(days=1)
    d = d.copy()
    d["is_national_holiday"] = d["date"].isin(national).astype("float32")
    d["holiday_tomorrow"]    = (d["date"] + one_day).isin(national).astype("float32")
    d["holiday_yesterday"]   = (d["date"] - one_day).isin(national).astype("float32")
    return d

def add_oil_lag_features(df: pd.DataFrame, oil: pd.DataFrame) -> pd.DataFrame:
    oil = oil.copy()
    oil["date"] = pd.to_datetime(oil["date"])
    s = oil.set_index("date")["dcoilwtico"]

    # Oil has no weekends/holidays -> make it daily, carry last known price forward
    days = pd.date_range(s.index.min(), df["date"].max(), freq="D")
    s = s.reindex(days).ffill()

    shifted = s.shift(16)
    feats = pd.DataFrame({
        "date": days,
        "oil_lag_16": shifted.values,
        "oil_rolling_mean_7": shifted.rolling(7).mean().values,
        "oil_rolling_mean_28": shifted.rolling(28).mean().values,
    })

    return df.merge(feats, on="date", how="left")

# add lag features
def add_lag_features(df : pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["store_nbr", "item_nbr", "date"]).reset_index(drop=True)
    df["log_sales"] = np.log1p(df["unit_sales"].astype("float32"))

    g = df.groupby(["store_nbr", "item_nbr"])["log_sales"]

    for lag in [16, 21, 28, 35]:
        df[f"lag_{lag}"] = g.shift(lag)

    df["rolling_mean_7"]  = g.transform(lambda s: s.shift(16).rolling(7).mean())
    df["rolling_mean_28"] = g.transform(lambda s: s.shift(16).rolling(28).mean())

    return df.copy().drop(columns=["log_sales"])

def log(df: pd.DataFrame, msg: str = "") -> pd.DataFrame:
    print(f"{msg}: {df.shape[0]:,} rows × {df.shape[1]} cols")
    return df

data_plus = (
    data
    .pipe(add_item_data, item_data)
    .pipe(log, "items added")
    .pipe(add_store_data, store_data)
    .pipe(log, "stores added")
    .pipe(add_holidays, holiday_data)
    .pipe(log, "holidays added")
    .pipe(add_oil_lag_features, oil_data)
    .pipe(log, "oil added")
    .pipe(add_lag_features)
    .pipe(log, "lags added")
)

# drop none rows
cols = ["lag_16", "lag_21","lag_28", "lag_35","rolling_mean_7", "rolling_mean_28"]
data_plus = data_plus.dropna(subset=cols, how="any")

data_plus.head()

items added: 89,139,096 rows × 8 cols
stores added: 89,139,096 rows × 12 cols
holidays added: 89,139,096 rows × 15 cols
oil added: 89,139,096 rows × 18 cols
lags added: 89,139,096 rows × 24 cols


,date,store_nbr,item_nbr,unit_sales,onpromotion,family,class,perishable,city,state,...,holiday_yesterday,oil_lag_16,oil_rolling_mean_7,oil_rolling_mean_28,lag_16,lag_21,lag_28,lag_35,rolling_mean_7,rolling_mean_28
43,2016-08-14,1,96995,0.0,NaN,GROCERY I,1093,0,Quito,Pichincha,...,0.0,41.54,42.278571,45.017500,0.0,0.0,0.0,0.0,0.0,0.0
44,2016-08-15,1,96995,0.0,NaN,GROCERY I,1093,0,Quito,Pichincha,...,0.0,41.54,42.011429,44.750357,0.0,0.0,0.0,0.0,0.0,0.0
45,2016-08-16,1,96995,0.0,NaN,GROCERY I,1093,0,Quito,Pichincha,...,0.0,41.54,41.744286,44.483214,0.0,0.0,0.0,0.0,0.0,0.0
46,2016-08-17,1,96995,0.0,NaN,GROCERY I,1093,0,Quito,Pichincha,...,0.0,40.05,41.408571,44.162857,0.0,0.0,0.0,0.0,0.0,0.0
47,2016-08-18,1,96995,0.0,NaN,GROCERY I,1093,0,Quito,Pichincha,...,0.0,39.50,41.028571,43.904643,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
data_plus.info()

<class 'pandas.DataFrame'>
Index: 79767504 entries, 43 to 89139095
Data columns (total 24 columns):
 #   Column               Dtype         
---  ------               -----         
 0   date                 datetime64[us]
 1   store_nbr            int8          
 2   item_nbr             int32         
 3   unit_sales           float32       
 4   onpromotion          object        
 5   family               str           
 6   class                int64         
 7   perishable           int64         
 8   city                 str           
 9   state                str           
 10  type                 str           
 11  cluster              int64         
 12  is_national_holiday  float32       
 13  holiday_tomorrow     float32       
 14  holiday_yesterday    float32       
 15  oil_lag_16           float64       
 16  oil_rolling_mean_7   float64       
 17  oil_rolling_mean_28  float64       
 18  lag_16               float32       
 19  lag_21               float32      

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import TargetEncoder, FunctionTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
import numpy as np

CAT_COLS = ["store_nbr", "family", "city", "state", "type", "cluster"]


def add_date_features(d):
    d = d.copy()
    # adds weekday and month feature
    d["date"] = pd.to_datetime(d["date"])
    d["weekday_num"] = d["date"].dt.day_of_week.astype("category")
    d["monthday_num"] = d["date"].dt.day.astype("int8")
    d["month_num"]    = d["date"].dt.month.astype("int8")
    return d.drop(columns="date")


def prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["onpromotion"] = df["onpromotion"].fillna(False).astype("int8")
    df["perishable"] = df["perishable"].astype("int8")

    df["dayofweek"] = df["date"].dt.dayofweek.astype("int8")
    df["day"]       = df["date"].dt.day.astype("int8")
    df["month"]     = df["date"].dt.month.astype("int8")

    df["is_national_holiday"] = df["is_national_holiday"].astype("int8")
    df["holiday_tomorrow"] = df["is_national_holiday"].astype("int8")
    df["holiday_yesterday"] = df["is_national_holiday"].astype("int8")

    df[CAT_COLS] = df[CAT_COLS].astype("category")
    return df

df = data_plus

(x_train, y_train, x_valid, y_valid) = split_train_set(df, "2017-07-31")

# Target-encode item_nbr and pass the other columns through unchanged
preprocess = ColumnTransformer(
    [("item_te", TargetEncoder(), ["item_nbr", "class"])],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")   # keeps store_nbr as 'category'

model = TransformedTargetRegressor(
    regressor=make_pipeline(
        FunctionTransformer(prepare_features),
        FunctionTransformer(add_date_features),
        preprocess,
        HistGradientBoostingRegressor(
            categorical_features=CAT_COLS,
            max_iter=1000, learning_rate=0.05, max_leaf_nodes=127,
            early_stopping=True,
        ),
    ),
    func=np.log1p,
    inverse_func=np.expm1,
    check_inverse=False,   # clipping can't be undone, so skip the round-trip check
)


model.fit(x_train, y_train)
pred = model.predict(x_valid)

/var/folders/hq/d3n2t3r135gfw07crm1cqnrh0000gn/T/ipykernel_83090/1226401454.py:25: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["dayofweek"] = df["date"].dt.dayofweek.astype("int8")
/var/folders/hq/d3n2t3r135gfw07crm1cqnrh0000gn/T/ipykernel_83090/1226401454.py:25: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["dayofweek"] = df["date"].dt.dayofweek.astype("int8")


In [ ]:
np.mean(pred)

np.float64(2.9683396939319397)

In [ ]:
# scoring

from sklearn.metrics import root_mean_squared_log_error

error = root_mean_squared_log_error(pred, y_valid)
print(f"Error for random sample set is: {error}")

Error for random sample set is: 0.5672854054636907


In [ ]:
# load test set and add lag
from pathlib import Path

test_data = load_data(EXTRACTED_PATH / "test.csv", dtype={"id": int})
test_data["unit_sales"] = None

# combine with previous history
combined = pd.concat([df[df["date"] >= "2017-06-01"], test_data], ignore_index=True)

# add zero sales and then lag features
combined_with_zeros = add_zero_sales(combined)
combined_with_lags = add_lag_features(combined_with_zeros)

combined_with_lags[combined_with_lags["id"] > 0].info()

<class 'pandas.DataFrame'>
Index: 3370464 entries, 76 to 20348927
Data columns (total 12 columns):
 #   Column           Dtype  
---  ------           -----  
 0   date             object 
 1   store_nbr        int64  
 2   item_nbr         int64  
 3   unit_sales       object 
 4   onpromotion      object 
 5   lag_16           float32
 6   lag_21           float32
 7   lag_28           float32
 8   lag_35           float32
 9   rolling_mean_7   float64
 10  rolling_mean_28  float64
 11  id               float64
dtypes: float32(4), float64(3), int64(2), object(3)
memory usage: 282.9+ MB


In [ ]:
# create csv
import os
from pathlib import Path

EXTRACTED_FOLDER = Path("./../data/raw") / "extracted"
RESULT_FOLDER = Path("./../data/results")

test_set = combined_with_lags[combined_with_lags["id"] > 0]

# make predictions
features = test_set.columns.drop(["unit_sales", "id"])
test_pred = model.predict(test_set[features])

# clip to 0 as minimum
test_pred_positive = np.clip(test_pred, 0, None)

submission_data = pd.DataFrame({
    "id": test_set["id"].to_numpy().astype("int64"),
    "unit_sales": np.asarray(test_pred_positive, dtype=float).ravel(),
})

write_result(submission_data, "lag_features_result_v2.csv")



# Scores

* test_split: "2017-07-31", data from ""2016-07-01" onwards: private: 0.55869 public: 0.59129

